In [ ]:
# 9.2 Exercises – Hotel Review Sentiment Analysis
#Claudia Galdamez

#I built a sentiment analysis model using the hotel reviews dataset.
#The goal of this assignment is to predict if a review is happy (0) or not happy (1) using machine learning.

In [1]:
#Import libraries
import pandas as pd
import numpy as np
import re
import string

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

I imported the necessary libraries.  
I am using Pandas for handling data, TF-IDF to convert text into numbers,  
and Logistic Regression for classification.

In [2]:
#Unzip file
import zipfile

with zipfile.ZipFile("/content/archive (2).zip", "r") as zip_ref:
    zip_ref.extractall("hotel_data")

In [4]:
#check what is in the file

import os

os.listdir("hotel_data")

['hotel-reviews.csv']

In [6]:
import pandas as pd

df = pd.read_csv("hotel_data/hotel-reviews.csv")
df.head()

,User_ID,Description,Browser_Used,Device_Used,Is_Response
0,id10326,The room was kind of clean but had a VERY stro...,Edge,Mobile,not happy
1,id10327,I stayed at the Crown Plaza April -- - April -...,Internet Explorer,Mobile,not happy
2,id10328,I booked this hotel through Hotwire at the low...,Mozilla,Tablet,not happy
3,id10329,Stayed here with husband and sons on the way t...,InternetExplorer,Desktop,happy
4,id10330,My girlfriends and I stayed here to celebrate ...,Edge,Tablet,not happy


The dataset contains hotel reviews and a column called "Is_Response".
The sentiment is currently written as text (happy / not happy).
I need to convert this into numbers.

In [7]:
#Lable encoding
#happy- 0
#not happy- 1

df["Is_Response"] = df["Is_Response"].map({"happy": 0, "not happy": 1})
df["Is_Response"].value_counts()

,count
Is_Response,
0,26521
1,12411


I converted the sentiment labels into numeric form.
0 represents happy reviews and 1 represents not happy reviews.


In [9]:
#Check columns
df.columns

Index(['User_ID', 'Description', 'Browser_Used', 'Device_Used', 'Is_Response'], dtype='object')

In [10]:
#Clean text
def clean_text(text):
    text = text.lower()
    text = re.sub(r"\d+", "", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    return text

df["cleaned_review"] = df["Description"].apply(clean_text)

I cleaned the text so the model can focus on meaningful words.

What was cleaned:
- converting everything to lowercase
- removing numbers
- removing punctuation


In [12]:
#Split into Train/validation and Test

X = df["cleaned_review"]
y = df["Is_Response"]

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)

X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

I split the data into:
- 70% training
- 15% validation
- 15% testing

The training set is used to teach the model and let it learn patterns in the reviews.

The validation set helps check how well the model is doing which allows me to make adjustments if needed.

The test set is used to evaluate final performance.

In [13]:

#Convert test to numbers
vectorizer = TfidfVectorizer(stop_words="english", max_features=5000)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_val_tfidf = vectorizer.transform(X_val)
X_test_tfidf = vectorizer.transform(X_test)

Machine learning models cannot understand text directly.
TF-IDF converts words into numeric features based on importance.
This allows the model to detect patterns in the reviews.

In [14]:
#Train Logistic regression model
model = LogisticRegression()
model.fit(X_train_tfidf, y_train)

LogisticRegression()

In [15]:
#Validation set test

val_predictions = model.predict(X_val_tfidf)

print("Validation Accuracy:", accuracy_score(y_val, val_predictions))

Validation Accuracy: 0.8784246575342466


In [16]:
#Final test
test_predictions = model.predict(X_test_tfidf)

print("Test Accuracy:", accuracy_score(y_test, test_predictions))
print("\nClassification Report:\n", classification_report(y_test, test_predictions))

Test Accuracy: 0.8785958904109589

Classification Report:
               precision    recall  f1-score   support

           0       0.89      0.93      0.91      3999
           1       0.84      0.76      0.80      1841

    accuracy                           0.88      5840
   macro avg       0.87      0.85      0.86      5840
weighted avg       0.88      0.88      0.88      5840



The model was able to correctly classify most of the reviews. The accuracy score shows it works well on reviews it hasn’t seen before. Using TF-IDF with Logistic Regression helped the model figure out whether a review was positive or negative. It’s not perfect, but it shows that machine learning can do a good job predicting customer sentiment from text.

##Citations

Scikit-learn. (n.d.). TfidfVectorizer. Retrieved from https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html

Scikit-learn. (n.d.). LogisticRegression. Retrieved from https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html

Scikit-learn. (n.d.). train_test_split. Retrieved from https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html

https://www.kaggle.com/datasets/harmanpreet93/hotelreviews?resource=download